# Graph Models and Hypothesis Testing (2026)

Utrecht Summer School on Network Science — Session 1


**The question for this morning:**

> *Is what I'm seeing in my network surprising, or is it what I'd get anyway?*

Every real network "has" clustering, hubs, groups. The question is never whether these
features exist — it's whether they need an explanation beyond chance. Today you'll learn a
**ladder of null models** that answers this, and by the end of the morning you'll apply it
to a real dataset (ideally your own).

This notebook follows the **Graph Models cheat sheet** step by step — the code you run here
is the same code on the sheet, so everything you practice this morning transfers directly
to your own data this afternoon and beyond.

Some of the libraries are difficult to install on some systems, so it is best to open this
notebook in Colab
[![Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jgarciab/NetworkScience/blob/main/Practicals/day2a_graph_models/Graph_models_and_hypothesis_testing_2026.ipynb)

**You cannot save changes in this notebook, you need to save a copy to your Google Drive:
`File` → `Save a copy in Drive`.**


## 0 — Setup (do this first, ~5 minutes)

We use [graph-tool](https://graph-tool.skewed.de/), which is not available in the standard
Colab environment, so we install it via conda. **Run the three cells below now** — the
install takes a few minutes, and the first cell will **restart the kernel** (that is
expected, not an error).


In [ ]:
!pip install -q condacolab
import condacolab
# Workaround issue with Python 3.12:
# condacolab.install()
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

*The kernel restarts after the cell above — wait for it, then continue below.
(If a crash message appears at this stage, just run the notebook again; this is related to
the kernel switch done by condacolab.)*


In [ ]:
!mamba install -q graph-tool

In [ ]:
# Workaround issue with Python 3.12
!mamba install -q scipy

In [ ]:
import graph_tool.all as gt
from graph_tool import topology, inference, generation, stats, correlations
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.stats import binomtest

## 1 — The null-model ladder

Two quick vignettes:

- An analyst finds that their **organisation's email network** has "unexpectedly high
  clustering" — colleagues of colleagues email each other a lot. Management concludes
  there are strong informal teams. *Is that conclusion justified?*
- In a **trade network**, one country sits in the middle of everything, and commentators
  conclude it is structurally special — a deliberate broker. *Or does it just trade a lot?*

In both cases the observation is real, but the *explanation* is only warranted if the
pattern exceeds what a suitably boring random baseline would produce. Choosing that
baseline is the whole game. We use an escalating ladder:

| Rung | Null model | What it holds fixed | Failing it tells you... |
|---|---|---|---|
| 1 | **Erdős–Rényi** | number of nodes & edges | "not maximally random" (almost everything fails this) |
| 2 | **Configuration model** | every node's degree | "there is structure beyond degree heterogeneity" |
| 3 | **DC-SBM** | degrees *and* group structure | "structure beyond blocks + degrees" → Wednesday's session |
| — | **BESTest** | a node attribute you have | "this attribute is (not) associated with the structure" |

**Each rung rules out a more sophisticated "boring" explanation.** Don't skip to the most
complex model — the point of the ladder is knowing *which* baseline your result actually
needs to beat.

The decision tree (also on your cheat sheet):

```
Do you have a specific count/degree you're questioning?
 └─ yes → Step 1 (ER / binomial test)

Is a global property (clustering, assortativity, degree distribution shape)
surprising relative to a random graph?
 └─ yes → Step 2 (configuration model null test)

Do you suspect group/block structure (even without knowing the groups)?
 └─ yes → Step 3 (DC-SBM) → Wednesday + Community Detection cheat sheet

Do you have a labelled attribute you want to test for relevance?
 └─ yes → BESTest
```

The rest of this notebook climbs this ladder on real data. First, a short gallery of the
network models themselves, so the words "random graph" and "configuration model" mean
something concrete.


## 2 — Baseline vocabulary: a fast gallery of network models

**This section is run-and-discuss, not typing.** All the code is complete — execute each
cell, look at the picture or plot, and answer the discussion prompt with your neighbour.
The graph-tool documentation for these models is
[here](https://graph-tool.skewed.de/static/doc/generation.html).

Real-world networks tend to exhibit three properties, and each model below captures some
but not all of them:

1. High clustering (closed triangles)
2. Short path lengths
3. Heavy-tailed degree distributions


### 2.1 Regular graphs and lattices

All nodes have (roughly) the same degree, arranged in a regular pattern.


In [ ]:
# generate square lattice
g = gt.lattice([10, 10])
pos = gt.sfdp_layout(g, cooling_step=0.95, epsilon=1e-2)
gt.graph_draw(g, pos=pos)

# generate ring lattice (each node connected to its 2 nearest neighbours on each side)
g = gt.circular_graph(20, 2)
pos = gt.sfdp_layout(g, cooling_step=0.95)
gt.graph_draw(g, pos=pos)

In [ ]:
print('Number of nodes', g.num_vertices())
print('Number of edges', g.num_edges())
print('Clustering coefficient', gt.local_clustering(g).fa.mean())
# Calculating the average path length exactly is expensive for large networks,
# so we use the pseudo diameter as an approximation
print('Average path length', gt.pseudo_diameter(g)[0])

**Discuss:** lattices have high clustering — which of the three real-world properties do
they clearly *fail* on? (Look at the path length relative to the number of nodes.)


### 2.2 Random (Erdős–Rényi) graphs

In the Erdős–Rényi model every pair of nodes has the same probability of being connected.
This is rung 1 of the ladder.


In [ ]:
g = gt.random_graph(100, lambda: np.random.poisson(10), directed=False)
gt.graph_draw(g)

In [ ]:
print('Number of nodes', g.num_vertices())
print('Number of edges', g.num_edges())
print('Clustering coefficient', gt.local_clustering(g).fa.mean())
print('Average path length', gt.pseudo_diameter(g)[0])

# degree distribution
degree_sequence = g.get_total_degrees(list(g.vertices()))
_ = plt.hist(degree_sequence, bins=10)
_ = plt.axvline(np.mean(degree_sequence), color='black')
_ = plt.title('Mean Degree = {:.2f}'.format(np.mean(degree_sequence)))

**Discuss:** compared to the lattice — what happened to clustering, and what happened
to path length? And how would you describe the *shape* of the degree distribution
(heavy-tailed or concentrated around the mean)?


### 2.3 Small-world networks: interpolating between the two

Watts–Strogatz-style small-world networks randomise a fraction $p$ of the links in a
regular ring, keeping the rest of the regular structure.

![image.png](https://drive.google.com/uc?id=1P8ewNjJuGlnIvZp1RKF2WCQwAFaZ8AL5)

The sweep below regenerates the network for 14 values of $p$ and tracks clustering and
diameter relative to the unrewired ring ($p=0$).


In [ ]:
n_networks = 14
rewire_probs = np.logspace(-4, 1, n_networks)
n_nodes = 1000
n_edges = 5000
g = gt.circular_graph(n_nodes, n_edges / n_nodes)

C0 = gt.global_clustering(g)[0]
L0 = gt.pseudo_diameter(g)[0]

clusts = np.empty(n_networks)
lengths = np.empty(n_networks)

for i in range(n_networks):
    g = gt.circular_graph(n_nodes, n_edges / n_nodes)
    p = rewire_probs[i]
    generation.random_rewire(g, n_iter=int(np.round(p * n_edges)), edge_sweep=False)
    clusts[i] = gt.global_clustering(g)[0] / C0
    lengths[i] = gt.pseudo_diameter(g)[0] / L0

plt.semilogx(rewire_probs, clusts, 'o', label='clustering')
plt.semilogx(rewire_probs, lengths, 'o', label='diameter')
plt.legend()
_ = plt.xlabel('Rewire probability (p)')

**Discuss:** what range of $p$ produces "realistic" networks — i.e. keeps high
clustering *and* short paths at the same time?


### 2.4 The Price model (preferential attachment)

The models so far all create relatively homogeneous degree distributions. The Price model
(Barabási–Albert, if undirected) grows a network where new nodes attach preferentially to
already-popular nodes — producing hubs and a heavy tail.


In [ ]:
g = gt.price_network(5000)

gt.graph_draw(g, pos=gt.sfdp_layout(g, cooling_step=0.99),
              vertex_fill_color=g.vertex_index, vertex_size=2,
              vcmap=mpl.cm.plasma,
              edge_pen_width=1)

In [ ]:
# degree distribution on log-log axes
in_hist = gt.vertex_hist(g, "total")

plt.figure(figsize=(6, 4))
plt.errorbar(in_hist[1][:-1], in_hist[0], fmt="o", yerr=0)
plt.gca().set_yscale("log")
plt.gca().set_xscale("log")
plt.gca().set_ylim(1e-1, 1e5)
plt.gca().set_xlim(0.8, 1e3)
plt.xlabel("$k$")
plt.ylabel("$NP(k)$")
plt.tight_layout()

The colours in the drawing indicate the order in which nodes were added — the oldest
nodes became the hubs. The straight line on log-log axes is the signature of a heavy tail.

**Why this matters for the ladder:** heavy-tailed degrees *on their own* can produce
apparent clustering, apparent communities, apparent core-periphery structure. So "my
network has hubs" is precisely the boring explanation that rung 2 is designed to control
for.


### 2.5 The configuration model: our rung-2 machine

The configuration model keeps every node's degree **exactly as observed** but rewires who
connects to whom, uniformly at random. One line of graph-tool turns any network into a
sample from its own configuration model:


In [ ]:
generation.random_rewire(g, model='configuration')  # degrees preserved, everything else scrambled
gt.graph_draw(g)

Same hubs, same degree histogram — but any *additional* structure (clustering,
communities, attribute alignment) has been destroyed. Repeating this many times gives a
null *distribution* for any statistic you care about. That recipe is the workhorse of the
rest of the morning, and of Wednesday too.


## 3 — Rung 1 in practice: "is this degree surprising?"

Time to climb the ladder on real data. Under ER with $n$ nodes and connection probability
$p$, a node's degree follows a binomial distribution — so "is this node's degree
surprisingly high?" is a textbook binomial test.

First, calibrate your intuition on a network where we *know* the answer, because we
generated it: an ER graph with 1000 nodes and mean degree 10.


In [ ]:
g = gt.random_graph(1000, lambda: np.random.poisson(10), directed=False)
degree_sequence = g.get_total_degrees(list(g.vertices()))

n = 1000    # number of nodes
c = 10      # mean degree
p = c / (n - 1)   # probability of a connection

# probability that a single node has at least 15 connections, by chance
test = binomtest(15, n - 1, p, alternative='greater')
test.pvalue

**Exercise 1a.** How many nodes in this network actually have at least 15 connections?
How many would you *expect* to have at least 15, given the p-value above?


In [ ]:
# How many nodes have degree >= 15?
n_at_least_15 = # insert code here (hint: degree_sequence)
print('Observed:', n_at_least_15)

# How many would you expect under ER? (hint: n and test.pvalue)
expected = # insert code here
print('Expected under ER:', expected)

### A real network: face-to-face contacts in a high school

The network below records face-to-face proximity interactions between high-school students
(measured with wearable sensors, [SocioPatterns](http://www.sociopatterns.org/), Marseille
2013). We take the interactions from the first hour of the school day, and collapse
repeated contacts between the same pair into a single edge.


In [ ]:
!wget https://networks.skewed.de/net/sp_high_school/files/proximity.gt.zst
g_proximity = gt.load_graph("proximity.gt.zst")

# interactions in the first hour of the dataset
g_prox_hour1 = gt.Graph(
    gt.GraphView(g_proximity, efilt=lambda e: g_proximity.ep.time[e] <= 1385982020 + 3600),
    prune=True)
# collapse repeated contacts between the same pair into one edge
gt.remove_parallel_edges(g_prox_hour1)

print('Nodes:', g_prox_hour1.num_vertices(), ' Edges:', g_prox_hour1.num_edges())

**Exercise 1b.** Plot the degree distribution of the hour-1 contact network. Then pick
the highest-degree node and test whether its degree is surprising under an ER null with the
same number of nodes and edges. (Cheat sheet, Step 1.)

*State your assumptions:* this is where real data gets messy. Some students have degree 0
in the first hour — should they count towards $n$? There is no single right answer, but
your test is only as meaningful as the null you chose.


In [ ]:
# 1. plot the degree distribution (histogram)
# insert code here

# 2. ER parameters for this network (cheat sheet, Step 1)
n = # insert code here
m = # insert code here
c = # insert code here     # mean degree
p = # insert code here

# 3. binomial test for the highest-degree node
k_max = # insert code here
test = binomtest(# insert code here)
print(test.pvalue)

**But hold on.** All we have shown is that this network is not *maximally* random.
ER shares edges out with no memory of node degrees, so almost any real network fails
against it — a rejection tells you almost nothing about *why*. This is the limitation of
rung 1, and the reason the ladder continues.


## 4 — Rung 2 in practice: the configuration-model null test

**The question:** once we fix each student's *number* of contacts, is there still extra
structure — for example clustering — that degree heterogeneity alone wouldn't produce?

**The recipe**:

1. compute your statistic on the observed network;
2. rewire with `model='configuration'` (degrees preserved), recompute the statistic;
3. repeat many times to build a null distribution;
4. the p-value is the fraction of null samples at least as extreme as the observation.

**Exercise 2.** Run this test for the mean local clustering coefficient of the hour-1
contact network. Plot the null distribution as a histogram with the observed value as a
vertical line, and compute the empirical p-value.


In [ ]:
observed = # insert code here   # mean local clustering of g_prox_hour1

g_rand = g_prox_hour1.copy()
n_samples = 200        # use 1000 for a final analysis; 200 keeps the class moving
stat_values = np.empty(n_samples)

for i in range(n_samples):
    # rewire g_rand preserving degrees, then store the statistic
    # insert code here

pvalue = # insert code here

# plot: histogram of stat_values + vertical line at observed
# insert code here

**Exercise 2b — swap the statistic.** The loop above is a *general* recipe: nothing
about it is specific to clustering. Rerun the test with degree assortativity
(`correlations.assortativity(g, 'total')[0]`) as the statistic. Do students with many
contacts preferentially meet other students with many contacts, beyond what their degrees
require?


In [ ]:
# same loop as Exercise 2, with correlations.assortativity(g, 'total')[0]
# as the statistic
# insert code here

## 5 — A five-minute aside: the friendship paradox

"Your friends have more friends than you do" — on average, in almost any network, because
high-degree nodes appear in many people's friend lists. It is a property of sampling, not
of you.

**Exercise 3.** Test whether the friendship paradox holds in the (full, all-days) proximity
network: for what proportion of students is their own degree higher than the average degree
of their contacts?


In [ ]:
# load the full proximity network and collapse repeated contacts into single edges
g_proximity_simple = g_proximity.copy()
gt.remove_parallel_edges(g_proximity_simple)
print(g_proximity_simple.num_vertices(), 'nodes,', g_proximity_simple.num_edges(), 'edges')

In [ ]:
# adjacency matrix
A = gt.adjacency(g_proximity_simple)
# vector of degrees
k = np.asarray(A.sum(axis=1)).ravel()
# mean degree of each node's neighbours (hint: A.dot(k) sums neighbour degrees)
friend_degrees = # insert code here
# proportion of nodes whose own degree exceeds their friends' average degree
popular_nodes = # insert code here
print('Proportion of students with more contacts than their average contact:', popular_nodes)

## 6 — The BESTest: "does this thing I know about my nodes actually matter?"

This is the most directly reusable technique of the morning. Almost everyone's own network
comes with node labels — department, team, country, category, school class. The obvious
question is whether that label is actually associated with the network structure, or
whether the alignment you think you see in the layout is in your head.

**The idea** ([Peel, Larremore & Clauset, *Science Advances* 2017](https://doi.org/10.1126/sciadv.1602548)):
treat the attribute as a *partition* of the nodes and measure how well it compresses the
network, using the entropy of a stochastic block model fixed to that partition. Lower
entropy = the partition explains the edge pattern better. Then compare against the entropy
of *random* partitions with the same number of groups: if the attribute does no better than
random labels, it is not associated with the structure.

This is the same null-distribution logic as Section 4 — we've just swapped "rewire the
network" for "shuffle the labels", and the statistic is now model-based.


In [ ]:
def random_entropy(g, n_groups):
    n = g.num_vertices()
    group_sizes = np.array(n_groups)                       # array of group sizes, e.g. np.bincount(b_observed.a)
    b = np.random.choice(len(group_sizes), size=n, p=group_sizes / group_sizes.sum())
    vprop_b = g.new_vertex_property("int64_t")
    for i in range(n):
        vprop_b[g.vertex(i)] = b[i]
    state = gt.BlockState(g, b=vprop_b)
    return state.entropy()

### The Lazega lawyers network

71 partners and associates of a US corporate law firm, with several relationship layers
(co-work, advice, friendship) and rich node attributes: which of the three **offices** they
sit in, their **practice** (litigation vs corporate), **status** (partner vs associate),
**law school**, and **gender**. We start with the co-work layer.


In [ ]:
!wget 'https://github.com/jgarciab/NetworkScience/raw/main/Data/law_firm.gt.zst'
g = gt.load_graph('law_firm.gt.zst')

# layer 1 = co-work
u = gt.GraphView(g, efilt=lambda e: g.ep.layer[e] == 1)
g1 = gt.Graph(u, prune=True)
print('Available node attributes:', list(g1.vp.keys()))

**Guided run — does the office explain who works with whom?** Run the three cells
below together with the room.


In [ ]:
attr_name = 'nodeOffice'
# Create a new vertex property map of type 'long' from the existing 'nodeOffice' property
# to ensure compatibility with gt.BlockState.
b = g1.new_vertex_property("int64_t", vals=g1.vp[attr_name])
state = gt.BlockState(g1, b=b)
test_value = state.entropy()
print('observed test statistic (entropy):', test_value)

In [ ]:
n_samples = 1000
n_groups = np.bincount(g1.vp[attr_name].a.astype(int))
null_entropy = np.empty(n_samples)
for i in range(n_samples):
    null_entropy[i] = random_entropy(g1, n_groups)

In [ ]:
plt.hist(null_entropy, bins=50, label='random partitions')
plt.axvline(test_value, color='black', label='office partition')
plt.legend()
plt.xlabel('SBM entropy (lower = partition explains structure better)')
plt.show()

pvalue = np.mean(null_entropy <= test_value)
print(f'p-value: {pvalue:.4f}')

**Discuss before scrolling on:** the observed entropy is *lower* than the null — what
does that mean in plain language? And why is the p-value computed with `<=` here, when in
Section 4 we used `>=`?


**Exercise 4 — which attribute explains the most?** Repeat the test for the other
categorical attributes: `nodePractice`, `nodeStatus`, `nodeLawSchool`, `nodeGender`.
Rank the attributes by how strongly they are associated with the co-work structure.

One catch: the raw entropies are **not comparable across attributes** — partitions with
different numbers of groups live on different scales. Compare each attribute against *its
own* null (e.g. via the p-value, or a z-score
$z = (H_{obs} - \bar H_{null}) / \sigma_{null}$ — more negative = more explanatory).


In [ ]:
attrs = ['nodeOffice', 'nodePractice', 'nodeStatus', 'nodeLawSchool', 'nodeGender']
n_samples = 1000

for attr_name in attrs:
    # observed entropy for this attribute's partition
    # insert code here

    # null distribution for this attribute's number of groups
    # insert code here

    # p-value and z-score
    # insert code here

**Exercise 5 — BESTest on the contact network.** The proximity network comes with node
attributes too (print `list(g_proximity_simple.vp.keys())` to see them). Test whether the
school **class** a student belongs to is associated with their face-to-face contacts. Given
Exercise 2, are you expecting a significant result?


In [ ]:
print('Available attributes:', list(g_proximity_simple.vp.keys()))

attr_name = 'class'
# NB: the class labels are strings -- you will need to map them to integers
# before passing them to gt.BlockState (BlockState needs an int vertex property)
# insert code here

**Keep the BESTest in your pocket.** It comes back on Wednesday with a second job:
after you *discover* communities with the SBM, you can validate them against an attribute
using exactly this test. One tool, two sessions.


## 7 — Practical time: the full ladder on a real, messy network

Now you run the whole argument end to end, with the cheat sheet as your guide and
instructors circulating. **Work in pairs. Pick ONE of the three datasets below** — each
comes with a naive claim of the kind you'll meet in the wild. Your job is to decide, rung
by rung, how much of the claim survives.

The pipeline (= the cheat sheet, top to bottom):

- **Step 0** — describe: nodes, edges, clustering, path length, degree distribution.
- **Step 1** — ER/binomial: is the most extreme degree surprising?
- **Step 2** — configuration null: does clustering (or another statistic) survive?
- **Step 3** — DC-SBM: `gt.minimize_blockmodel_dl(g, state_args={'deg_corr': True})` —
  is there block structure beyond degrees? (A preview of Wednesday: just fit it and look
  at the number of groups it finds.)
- **BESTest** — if the dataset has a node attribute, test whether it matters.

**As soon as your own data is ready, switch to it** — same cheat sheet, and the loader you
smoke-tested in Section 0b (full scaffold in the BYOD starter notebook). There is no
"own-data moment"; just move when you're ready.

One helper first — real datasets often store attributes as strings, and `BlockState` needs
integers:


In [ ]:
def categorical_to_int(g, prop_name):
    """Map a categorical vertex property to an int vertex property (0..K-1).
    Returns the int property and the {category: int} mapping."""
    labels = [g.vp[prop_name][v] for v in g.vertices()]
    cats = sorted(set(labels))
    mapping = {c: i for i, c in enumerate(cats)}
    b = g.new_vertex_property('int64_t') # Changed 'int' to 'int64_t'
    for v in g.vertices():
        b[v] = mapping[g.vp[prop_name][v]]
    return b, mapping

### Dataset A — Computer science faculty hiring

*Who hires whose PhD graduates as faculty* among 206 US/Canadian computer science
departments ([Clauset, Arbesman & Larremore, 2015](https://doi.org/10.1126/sciadv.1400005)).
A directed edge $i \to j$ means a PhD from $i$ became faculty at $j$. Node attributes
include a prestige score (`pi`) and geographic `Region`.

**The naive claim:** *"A handful of elite departments are structurally special hubs of the
hiring system."* Special compared to what — is it more than a heavy-tailed degree
distribution? And is geography (`Region`) associated with who hires from whom?


In [ ]:
g_fh = gt.collection.ns["faculty_hiring/computer_science"]
print(g_fh)
print('Node attributes:', list(g_fh.vp.keys()))
print('Edge attributes:', list(g_fh.ep.keys()))

# for the ladder we work with a simple undirected version
g_fhu = g_fh.copy()
g_fhu.set_directed(False)
gt.remove_parallel_edges(g_fhu)
gt.remove_self_loops(g_fhu)   # self-hires exist! decide for yourself if they belong
print('Simple undirected version:', g_fhu.num_vertices(), 'nodes,',
      g_fhu.num_edges(), 'edges')

In [ ]:
# Step 0 — describe
# insert code here

In [ ]:
# Step 1 — is the top department's degree surprising under ER?
# insert code here

In [ ]:
# Step 2 — configuration null (clustering, or another statistic you find interesting)
# insert code here

In [ ]:
# Step 3 — DC-SBM preview: how many groups does it find?
# insert code here

In [ ]:
# BESTest — is Region associated with the hiring structure?
# (hint: categorical_to_int)
# insert code here

### Dataset B — International food trade

Country-to-country trade from the FAO multiplex dataset
([De Domenico et al., 2015](https://doi.org/10.1038/ncomms7864)): 214 countries, with a
separate layer for each of ~360 products. Pick a product you like and extract its layer.

**The naive claim:** *"Country X sits at the centre of this product's trade — it is a
structurally special broker."* (Pick your own X once you see the degree ranking.)

**Note what's missing:** this dataset has *no categorical node attribute* — so look at your
decision tree: the BESTest branch simply does not apply. Part of using the ladder on your
own data is recognising which rungs your data can support.


In [ ]:
g_tr = gt.collection.ns["fao_trade"]
print(g_tr)
print('Node attributes:', list(g_tr.vp.keys()))
print('Edge attributes:', list(g_tr.ep.keys()))
print('Graph attributes:', list(g_tr.gp.keys()))

In [ ]:
# extract one product layer -- check the printed edge-attribute names above for the
# layer property (and adjust the property/value names below if they differ)
layer_prop = g_tr.ep['layer']
layer_id = 1   # pick a product; if a list of layer names exists in g_tr.gp, use it
g_layer = gt.Graph(gt.GraphView(g_tr, efilt=lambda e: layer_prop[e] == layer_id),
                   prune=True)
g_layer.set_directed(False)
gt.remove_parallel_edges(g_layer)
print(g_layer.num_vertices(), 'countries,', g_layer.num_edges(), 'trade links')

In [ ]:
# Steps 0-3: is your country X more than a big degree?
# insert code here

### Now try your own data



In the last part of this morning you can switch to **your own network data**.

1. Upload your edge-list CSV to a folder in your Google Drive (see `BRING_YOUR_OWN_DATA.md`
   for the format).
2. Run the cell below and authorize access.
3. Edit `my_data_path` to point at your file and check that it prints **File found**.

No data of your own (yet)? Run the cell anyway to test the Drive mount, then move on.


In [ ]:
from google.colab import drive
drive.mount('/drive')

import os
# --- edit this path to point at your own file ---
my_data_path = '/drive/MyDrive/network_summer_school/my_edgelist.csv'

if os.path.exists(my_data_path):
    print('File found — you are ready for the own-data practical.')
else:
    print('File NOT found at', my_data_path)
    print('Check the folder name and file name (case-sensitive), or ask an instructor now.')

# Alternative if you prefer not to mount Drive: uncomment the two lines below to
# upload a file directly into the Colab session instead (it disappears when the
# session ends, so you would need to re-upload tomorrow).
# from google.colab import files
# uploaded = files.upload()

## 8 — Where this goes next

**Today's argument in one line:** a network property is only meaningful relative to a null
model, and the ladder (ER → configuration → DC-SBM, with the BESTest for attributes) tells
you which "boring" explanation you've ruled out so far.

**Wednesday** starts exactly where rung 3 left off: *my network looks like it has groups —
is that real, and what are they?* Same null-test recipe, same BESTest, new question.
